# Data integrity check

Per-day and per-instrument sanity checks on the bundled `data2/`
corpus. Verifies the three invariants the loader / pricer rely on:

1. **Every yield-curve date has a short-rate entry** — so we can read
   `r0` as the anchor for the decoder shift.
2. **Every yield-curve date has at least one futures quote** — so the
   joint-loss branch always has something to fit.
3. **Every futures contract has a non-empty deliverable basket** —
   so `BatchedFuturesTarget.deliverable_ids_flat` is well-defined.
4. *(Plus:)* **every deliverable bond appears in `bond_meta.csv`** —
   otherwise `BondMetadataStore.get_bond_features` would raise on
   the corresponding date.

Run end-to-end. Each section prints a short summary and (on failure)
the offending dates / IDs.


In [ ]:
# Bootstrap: make `import src.*` work whether we launch from the repo
# root or from notebooks/.
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('repo root =', ROOT)


In [ ]:
import pandas as pd

DATA = ROOT / 'data2'
yc  = pd.read_csv(DATA / 'yield_curves.csv', parse_dates=['Date']).set_index('Date').sort_index()
sr  = pd.read_csv(DATA / 'short_rate.csv',   parse_dates=['Date']).set_index('Date').sort_index()
fq  = pd.read_csv(DATA / 'futures.csv',      parse_dates=['Date'])
fex = pd.read_csv(DATA / 'futures_expirations.csv', parse_dates=['DLV_Date'])
fdv = pd.read_csv(DATA / 'futures_dlv.csv')
bm  = pd.read_csv(DATA / 'bond_meta.csv')

yc.index = yc.index.normalize(); sr.index = sr.index.normalize()
fq['Date'] = fq['Date'].dt.normalize()

print(f'yield_curves : {len(yc):>6}  dates,  {yc.index.min().date()} -> {yc.index.max().date()}')
print(f'short_rate   : {len(sr):>6}  dates,  {sr.index.min().date()} -> {sr.index.max().date()}')
print(f'futures      : {len(fq):>6}  quote rows, {fq["Ticker"].nunique()} tickers')
print(f'expirations  : {len(fex):>6}  rows')
print(f'deliverables : {len(fdv):>6}  rows, {fdv["Bond_ID"].nunique()} unique bonds')
print(f'bond_meta    : {len(bm):>6}  bonds')


## Check 1 — every yield-curve date has a short-rate entry

In [ ]:
missing_sr = yc.index.difference(sr.index)
print(f'YC dates without a short-rate entry: {len(missing_sr)}')
if len(missing_sr):
    print('  first 10:')
    for d in missing_sr[:10]:
        print('   ', d.date())
    print('   …')
    print('NOTE: the loader uses reindex(method="ffill"), so this fails only at the very start.')
else:
    print('OK — every YC date has a short rate.')


## Check 2 — every yield-curve date has at least one futures quote

In [ ]:
# A futures quote is a non-NaN row in the wide pivot.
f_by_date = fq.dropna(subset=['Price']).groupby('Date').size()
yc_dates_with_fut = yc.index.intersection(f_by_date.index)
yc_dates_no_fut   = yc.index.difference(f_by_date.index)

print(f'YC dates WITH at least one futures quote   : {len(yc_dates_with_fut):>6} / {len(yc)}')
print(f'YC dates WITHOUT any futures quote          : {len(yc_dates_no_fut):>6} / {len(yc)}')
if len(yc_dates_no_fut):
    head = yc_dates_no_fut.min().date() if len(yc_dates_no_fut) else None
    tail = yc_dates_no_fut.max().date() if len(yc_dates_no_fut) else None
    print(f'  span: {head} .. {tail}')
    print('  first 10:')
    for d in yc_dates_no_fut[:10]:
        print('   ', d.date())


## Check 3 — every futures contract has a non-empty deliverable basket

i.e. every `Ticker` that appears in `futures.csv` (or `futures_expirations.csv`) has rows in `futures_dlv.csv`.

In [ ]:
tickers_q   = set(fq['Ticker'].unique())
tickers_exp = set(fex['Ticker'].unique())
tickers_dlv = set(fdv['Ticker'].unique())

missing_basket_from_q   = sorted(tickers_q   - tickers_dlv)
missing_basket_from_exp = sorted(tickers_exp - tickers_dlv)
missing_exp_from_q      = sorted(tickers_q   - tickers_exp)

print(f'tickers seen in futures.csv             : {len(tickers_q)}')
print(f'tickers seen in futures_expirations.csv : {len(tickers_exp)}')
print(f'tickers seen in futures_dlv.csv         : {len(tickers_dlv)}')
print()
print(f'TICKERS PRICED BUT WITHOUT a basket     : {len(missing_basket_from_q)}')
for t in missing_basket_from_q[:20]:
    print('   ', t)
if len(missing_basket_from_q) > 20:
    print('   …')
print()
print(f'tickers expired but without a basket    : {len(missing_basket_from_exp)}')
print(f'tickers priced but without an expiration: {len(missing_exp_from_q)}')


## Check 4 — every deliverable bond is in `bond_meta.csv`

In [ ]:
bond_ids_dlv  = set(fdv['Bond_ID'].astype(str).unique())
bond_ids_meta = set(bm['Bond_ID'].astype(str).unique())
missing_bonds = sorted(bond_ids_dlv - bond_ids_meta)

print(f'deliverable bonds                      : {len(bond_ids_dlv)}')
print(f'bonds in bond_meta.csv                 : {len(bond_ids_meta)}')
print(f'bonds REFERENCED but missing metadata  : {len(missing_bonds)}')
for b in missing_bonds[:20]:
    print('   ', b)
if len(missing_bonds) > 20:
    print('   …')


## Check 5 — the live snapshot the loader emits is consistent

Sanity-check that for a random YC date, the `MarketDataLoader.get_snapshot()`
output respects: deliverable IDs in `futures.deliverable_ids_flat` line up with
`bonds_metadata.ids` (post-dedup).

In [ ]:
from datetime import datetime
from src.configs import DataLoaderCfg
from src.dataloaders import MarketDataLoader

dl = MarketDataLoader(DataLoaderCfg(
    data_path=str(DATA),
    start_date=datetime(2021, 1, 1),
    end_date=datetime(2022, 12, 31),
    max_maturity=10,
    enable_yield=True, enable_short_rate=True, enable_futures=True,
))

n_check = 0
n_failures = 0
for d in dl.calendar.dates[::50]:           # 1 in 50 sample
    snap = dl.get_snapshot(d)
    if snap.futures is None:
        continue
    n_check += 1
    # Every slot in deliverable_ids_flat should be present in bonds_metadata.ids
    missing = set(snap.futures.deliverable_ids_flat) - set(snap.bonds_metadata.ids)
    if missing:
        n_failures += 1
        print(f'  {d.date()}: {len(missing)} deliverable IDs missing from bonds_metadata')
print(f'\n{n_check} dates inspected,  {n_failures} failures.')


## Summary

The four checks above cover every assumption the loader / pricer
makes on the CSV corpus. If any of them prints a non-zero failure
count, **the corresponding date range should be trimmed out of the
training window** via `DataLoaderCfg.start_date` / `end_date`
before launching a joint YC + futures run.